In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel



In [2]:
df = pd.read_csv("../data/potential-talents - Aspiring human resources - seeking human resources.csv")

df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


In [3]:
df = df.drop("fit",axis=1)

In [4]:
df.head()

,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+


In [5]:
print(df["connection"].unique())

['85' '500+ ' '44' '1' '61' '2' '390' '57' '82' '5' '7' '16' '212' '409'
 '52' '455' '174' '268' '50' '4' '40' '18' '349' '155' '39' '64' '9' '415'
 '19' '71' '48' '103' '49']


In [6]:
df['connection'] = df['connection'].replace("500+ ","500")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          104 non-null    int64 
 1   job_title   104 non-null    object
 2   location    104 non-null    object
 3   connection  104 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.4+ KB


In [8]:
df = df.drop_duplicates(subset=['job_title'])

In [9]:
df.head()


,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500


In [10]:
df.columns

Index(['id', 'job_title', 'location', 'connection'], dtype='object')

In [11]:
df[df["location"].str.contains(r"[^\x00-\x7F]", regex=True, na=False)]

,id,job_title,location,connection
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500
77,78,Human Resources Generalist at Schwan's,Amerika Birleşik Devletleri,500
93,94,Seeking Human Resources Opportunities. Open t...,Amerika Birleşik Devletleri,415


In [12]:
df['location'] = df['location'].str.replace("ü","u")
df['location'] = df['location'].str.replace("ş","s")
df['location'] = df['location'].str.replace("İ","I")

In [14]:
df["all_data"] = (
    df[["job_title", "location", "connection"]]
    .astype(str)
    .agg(" ".join, axis=1)
)

In [15]:
nlp = spacy.load("en_core_web_sm")

df['all_data'] = df['all_data'].astype("str").str.strip()

In [14]:
#!python -m spacy download en_core_web_sm

The idea is to build a function and apply it to each row in the job title column. The function tokenizes the text, removes punctuation and stop words, converts the words to lowercase, lemmatizes them, and returns the cleaned words as a list for each row in the job_title and return the join str becasue each model will use its own tokenization method.

In [16]:
def clean_title(x):
    doc = nlp(x)

    clean_text = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            clean_text.append(token.lemma_.lower())

    return " ".join(clean_text)

In [17]:
df["clean_job_title"] = df["all_data"].apply(clean_title)

In [18]:
df['clean_job_title']

0      2019 c.t. bauer college business graduate magn...
1      native english teacher epik english program ko...
2      aspiring human resources professional raleigh ...
3      people development coordinator ryan denton tex...
4      advisory board member celal bayar university i...
5      aspiring human resources specialist greater ne...
6      student humber college aspiring human resource...
7        hr senior specialist san francisco bay area 500
9      seek human resources hris generalist positions...
10     student chapman university lake forest califor...
11     svp chro marketing communications csr officer ...
12     human resources coordinator intercontinental b...
26     aspiring human resources management student se...
27     seek human resources opportunities chicago ill...
65     experienced retail manager aspire human resour...
66     human resources staffing recruiting profession...
67     human resources specialist luxottica greater n...
68     director human resources

In [19]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

BoW  is defined as how many times does the word appear?
TF-IDF is defined as how important/distinctive is this word compared with the rest of the dataset?

In [22]:
# BOW method 
# You fit and transform on the column
# you only transform the y1 and y2
vectorizer_bow = CountVectorizer()
X = vectorizer_bow.fit_transform(df['clean_job_title'])
y1 = vectorizer_bow.transform([measure_rank_1])
y2 = vectorizer_bow.transform([measure_rank_2])

In [23]:
score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

In [82]:
df["similarity_1_bow"] = score_1.flatten()
df[df["similarity_1_bow"]>0.4][["job_title", "similarity_1_bow"]].sort_values(
    by="similarity_1_bow",
    ascending=False
).head(10)

,job_title,similarity_1_bow
27,Seeking Human Resources Opportunities,0.749111
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.747322
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.735730
66,"Human Resources, Staffing and Recruiting Professional",0.720959
26,Aspiring Human Resources Management student seeking an internship,0.720783
65,Experienced Retail Manager and aspiring Human Resources Professional,0.719675
6,Student at Humber College and Aspiring Human Resources Generalist,0.719633
83,Human Resources professional for the world leader in GIS software,0.715453
103,Director Of Administration at Excellence Logging,0.707068
5,Aspiring Human Resources Specialist,0.702381


In [83]:
df["similarity_2_bow"] = score_2.flatten()
df[df["similarity_2_bow"]>0.4][["job_title", "similarity_2_bow"]].sort_values(
    by="similarity_2_bow",
    ascending=False
).head(10)

,job_title,similarity_2_bow
27,Seeking Human Resources Opportunities,0.801287
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.702566
66,"Human Resources, Staffing and Recruiting Professional",0.700950
9,Seeking Human Resources HRIS and Generalist Positions,0.698048
98,Seeking Human Resources Position,0.684404
91,Seeking employment opportunities within Customer Service or Patient Care,0.683962
103,Director Of Administration at Excellence Logging,0.683022
3,People Development Coordinator at Ryan,0.679889
73,Human Resources Professional,0.679089
88,Director Human Resources at EY,0.666376


In [26]:
# Tfidf method
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df['clean_job_title'])
y1 = vectorizer_tfidf.transform([measure_rank_1])
y2 = vectorizer_tfidf.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

In [81]:
df["similarity_1_tfidf"] = score_1.flatten()
df[df["similarity_1_tfidf"]>0.3][["job_title", "similarity_1_tfidf"]].sort_values(
    by="similarity_1_tfidf",
    ascending=False
).head(10)

,job_title,similarity_1_tfidf
27,Seeking Human Resources Opportunities,0.749111
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.747322
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.735730
66,"Human Resources, Staffing and Recruiting Professional",0.720959
26,Aspiring Human Resources Management student seeking an internship,0.720783
65,Experienced Retail Manager and aspiring Human Resources Professional,0.719675
6,Student at Humber College and Aspiring Human Resources Generalist,0.719633
83,Human Resources professional for the world leader in GIS software,0.715453
103,Director Of Administration at Excellence Logging,0.707068
5,Aspiring Human Resources Specialist,0.702381


In [80]:
df["similarity_2_tfidf"] = score_2.flatten()
df[df["similarity_2_tfidf"]>0.3][["job_title", "similarity_2_tfidf"]].sort_values(
    by="similarity_2_tfidf",
    ascending=False
).head(10)

,job_title,similarity_2_tfidf
27,Seeking Human Resources Opportunities,0.801287
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.702566
66,"Human Resources, Staffing and Recruiting Professional",0.700950
9,Seeking Human Resources HRIS and Generalist Positions,0.698048
98,Seeking Human Resources Position,0.684404
91,Seeking employment opportunities within Customer Service or Patient Care,0.683962
103,Director Of Administration at Excellence Logging,0.683022
3,People Development Coordinator at Ryan,0.679889
73,Human Resources Professional,0.679089
88,Director Human Resources at EY,0.666376


In [30]:
#%pip install gensim

model = api.load('glove-wiki-gigaword-100')

Each word has a GloVe vector → average all word vectors → get one vector representing the entire job title.

In [31]:
# apply has the tendency to work on the rows automatically 
def get_title_vector(row):
    words = row.split()

    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    return np.mean(vectors, axis=0)

In [32]:
df['vector_for_modeling'] = df["clean_job_title"].apply(get_title_vector)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)

In [33]:
X_glove = np.vstack(df["vector_for_modeling"])
# to make it work you need to use vstack function
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_glove
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_glove
)


In [79]:
df["similarity_1_glove"] = score_1.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_1_glove"]>0.3][["job_title", "similarity_1_glove"]].sort_values(
    by="similarity_1_glove",
    ascending=False
).head(10)

,job_title,similarity_1_glove
27,Seeking Human Resources Opportunities,0.749111
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.747322
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.735730
66,"Human Resources, Staffing and Recruiting Professional",0.720959
26,Aspiring Human Resources Management student seeking an internship,0.720783
65,Experienced Retail Manager and aspiring Human Resources Professional,0.719675
6,Student at Humber College and Aspiring Human Resources Generalist,0.719633
83,Human Resources professional for the world leader in GIS software,0.715453
103,Director Of Administration at Excellence Logging,0.707068
5,Aspiring Human Resources Specialist,0.702381


In [35]:
df["similarity_2_glove"] = score_2.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_2_glove"]>0.3][["job_title", "similarity_2_glove"]].sort_values(
    by="similarity_2_glove",
    ascending=False
).head(10)




,job_title,similarity_2_glove
27,Seeking Human Resources Opportunities,0.874183
72,"Aspiring Human Resources Manager, seeking inte...",0.871159
9,Seeking Human Resources HRIS and Generalist Po...,0.863006
93,Seeking Human Resources Opportunities. Open t...,0.862108
88,Director Human Resources at EY,0.826413
87,Human Resources Management Major,0.825759
73,Human Resources Professional,0.820881
83,Human Resources professional for the world lea...,0.809376
26,Aspiring Human Resources Management student se...,0.808565
75,Aspiring Human Resources Professional | Passio...,0.801996


In [56]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def get_title_vector_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
# pt is py_torch, truncation for fitting what the model allow in size, padding makes the size the same
    with torch.no_grad():
        outputs = bert(**inputs)


    vector = outputs.last_hidden_state.mean(dim=1)

    return vector.squeeze().numpy()
# make the tensor a numpy array with one dimension 
df['bert_for_modeling'] = df["clean_job_title"].apply(get_title_vector_bert)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [77]:
X_bert = np.vstack(df["bert_for_modeling"])

y1_vector = get_title_vector_bert(measure_rank_1)
y2_vector = get_title_vector_bert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bert
)

df["similarity_1_bert"] = score_1.flatten()

df[df["similarity_1_bert"]>0.3][["job_title", "similarity_1_bert"]].sort_values(
    by="similarity_1_bert",
    ascending=False
).head(10)


,job_title,similarity_1_bert
27,Seeking Human Resources Opportunities,0.749111
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.747322
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.735730
66,"Human Resources, Staffing and Recruiting Professional",0.720959
26,Aspiring Human Resources Management student seeking an internship,0.720783
65,Experienced Retail Manager and aspiring Human Resources Professional,0.719675
6,Student at Humber College and Aspiring Human Resources Generalist,0.719633
83,Human Resources professional for the world leader in GIS software,0.715453
103,Director Of Administration at Excellence Logging,0.707068
5,Aspiring Human Resources Specialist,0.702381


In [78]:
df["similarity_2_bert"] = score_2.flatten()

df[df["similarity_2_bert"]>0.3][["job_title", "similarity_2_bert"]].sort_values(
    by="similarity_2_bert",
    ascending=False
).head(10)

,job_title,similarity_2_bert
27,Seeking Human Resources Opportunities,0.801287
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.702566
66,"Human Resources, Staffing and Recruiting Professional",0.700950
9,Seeking Human Resources HRIS and Generalist Positions,0.698048
98,Seeking Human Resources Position,0.684404
91,Seeking employment opportunities within Customer Service or Patient Care,0.683962
103,Director Of Administration at Excellence Logging,0.683022
3,People Development Coordinator at Ryan,0.679889
73,Human Resources Professional,0.679089
88,Director Human Resources at EY,0.666376


In [59]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_title_vector_sbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = sbert(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Mean pooling, ignoring padding tokens
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["sbert_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_sbert
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [60]:
s_bert = np.vstack(df["sbert_for_modeling"])

y1_vector = get_title_vector_sbert(measure_rank_1)
y2_vector = get_title_vector_sbert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    s_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    s_bert
)

df["similarity_1_sbert"] = score_1.flatten()

df[df["similarity_1_sbert"]>0.4][["all_data", "similarity_1_sbert"]].sort_values(
    by="similarity_1_sbert",
    ascending=False
).head(10)

,all_data,similarity_1_sbert
2,Aspiring Human Resources Professional Raleigh-...,0.819170
5,Aspiring Human Resources Specialist Greater Ne...,0.788344
99,Aspiring Human Resources Manager | Graduating ...,0.775312
72,"Aspiring Human Resources Manager, seeking inte...",0.731115
98,"Seeking Human Resources Position Las Vegas, Ne...",0.710243
81,Aspiring Human Resources Professional | An ene...,0.704329
6,Student at Humber College and Aspiring Human R...,0.702629
26,Aspiring Human Resources Management student se...,0.701894
75,Aspiring Human Resources Professional | Passio...,0.699898
27,"Seeking Human Resources Opportunities Chicago,...",0.670924


In [61]:
df["similarity_2_sbert"] = score_2.flatten()

df[df["similarity_2_sbert"]>0.4][["all_data", "similarity_2_sbert"]].sort_values(
    by="similarity_2_sbert",
    ascending=False
).head(10)

,all_data,similarity_2_sbert
98,"Seeking Human Resources Position Las Vegas, Ne...",0.761732
2,Aspiring Human Resources Professional Raleigh-...,0.734525
5,Aspiring Human Resources Specialist Greater Ne...,0.717399
27,"Seeking Human Resources Opportunities Chicago,...",0.717316
9,Seeking Human Resources HRIS and Generalist Po...,0.699338
100,Human Resources Generalist at Loparex Raleigh-...,0.671056
99,Aspiring Human Resources Manager | Graduating ...,0.661074
72,"Aspiring Human Resources Manager, seeking inte...",0.658575
81,Aspiring Human Resources Professional | An ene...,0.653204
75,Aspiring Human Resources Professional | Passio...,0.640741


In [62]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [63]:
X_e5 = np.vstack(df["e5_for_modeling"])

y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)

In [64]:
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

df["similarity_1_5e"] = score_1.flatten()

df[df["similarity_1_5e"]>0.3][["all_data", "similarity_1_5e"]].sort_values(
    by="similarity_1_5e",
    ascending=False
).head(10)

,all_data,similarity_1_5e
5,Aspiring Human Resources Specialist Greater Ne...,0.900310
72,"Aspiring Human Resources Manager, seeking inte...",0.894750
99,Aspiring Human Resources Manager | Graduating ...,0.890414
2,Aspiring Human Resources Professional Raleigh-...,0.886353
26,Aspiring Human Resources Management student se...,0.878086
75,Aspiring Human Resources Professional | Passio...,0.876692
81,Aspiring Human Resources Professional | An ene...,0.864373
71,Business Management Major and Aspiring Human R...,0.863725
98,"Seeking Human Resources Position Las Vegas, Ne...",0.863497
27,"Seeking Human Resources Opportunities Chicago,...",0.860379


In [65]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_e5
)

df["similarity_2_5e"] = score_2.flatten()

df[df["similarity_2_5e"]>0.3][["all_data", "similarity_2_5e"]].sort_values(
    by="similarity_2_5e",
    ascending=False
).head(10)

,all_data,similarity_2_5e
27,"Seeking Human Resources Opportunities Chicago,...",0.892398
98,"Seeking Human Resources Position Las Vegas, Ne...",0.890624
72,"Aspiring Human Resources Manager, seeking inte...",0.882211
9,Seeking Human Resources HRIS and Generalist Po...,0.876815
99,Aspiring Human Resources Manager | Graduating ...,0.872307
93,Seeking Human Resources Opportunities. Open t...,0.872267
5,Aspiring Human Resources Specialist Greater Ne...,0.865797
26,Aspiring Human Resources Management student se...,0.865030
75,Aspiring Human Resources Professional | Passio...,0.860166
74,"Nortia Staffing is seeking Human Resources, Pa...",0.859875


In [69]:
df['all_data'][27]

'Seeking Human Resources Opportunities Chicago, Illinois 390'

In [43]:
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

bge = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


def get_title_vector_bge(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = bge(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy() 


df["bge_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_bge
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [44]:
X_bge = np.vstack(df["bge_for_modeling"])

y1_vector = get_title_vector_bge(measure_rank_1)
y2_vector = get_title_vector_bge(measure_rank_2)

In [49]:
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bge
)

df["similarity_1_bge"] = score_1.flatten()
# Chain the filtering and sorting together
df[df["similarity_1_bge"] > 0.3][["all_data", "similarity_1_bge"]].sort_values(
    by="similarity_1_bge",
    ascending=False
).head(10)

,all_data,similarity_1_bge
2,Aspiring Human Resources Professional Raleigh-...,0.786999
72,"Aspiring Human Resources Manager, seeking inte...",0.783562
75,Aspiring Human Resources Professional | Passio...,0.780927
5,Aspiring Human Resources Specialist Greater Ne...,0.778894
26,Aspiring Human Resources Management student se...,0.767548
99,Aspiring Human Resources Manager | Graduating ...,0.755488
27,"Seeking Human Resources Opportunities Chicago,...",0.754207
71,Business Management Major and Aspiring Human R...,0.747347
81,Aspiring Human Resources Professional | An ene...,0.743157
6,Student at Humber College and Aspiring Human R...,0.731464


In [50]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bge
)

df["similarity_2_bge"] = score_2.flatten()

df[df["similarity_2_bge"] > 0.3][["all_data", "similarity_2_bge"]].sort_values(
    by="similarity_2_bge",
    ascending=False
).head(10)

,all_data,similarity_2_bge
27,"Seeking Human Resources Opportunities Chicago,...",0.828740
9,Seeking Human Resources HRIS and Generalist Po...,0.774525
72,"Aspiring Human Resources Manager, seeking inte...",0.768068
98,"Seeking Human Resources Position Las Vegas, Ne...",0.761926
2,Aspiring Human Resources Professional Raleigh-...,0.758372
93,Seeking Human Resources Opportunities. Open t...,0.736612
5,Aspiring Human Resources Specialist Greater Ne...,0.730112
75,Aspiring Human Resources Professional | Passio...,0.729630
66,"Human Resources, Staffing and Recruiting Profe...",0.724346
74,"Nortia Staffing is seeking Human Resources, Pa...",0.721670


Among the evaluated embedding approaches, E5 produced the most relevant job-title rankings. This is consistent with E5's retrieval-oriented training, which is specifically designed to represent semantically related texts close together in the embedding space.

In [71]:
comparison_1 = pd.DataFrame({
    "BOW": df.nlargest(10,"similarity_1_bow")["all_data"].values,
    "TF-IDF": df.nlargest(10, "similarity_1_tfidf")["all_data"].values,
    "GloVe": df.nlargest(10, "similarity_1_glove")["all_data"].values,
    "BERT": df.nlargest(10, "similarity_1_bert")["all_data"].values,
    "SBERT": df.nlargest(10,"similarity_1_sbert")["all_data"].values,
    "E5": df.nlargest(10, "similarity_1_5e")["all_data"].values,
    "BGE": df.nlargest(10, "similarity_1_bge")["all_data"].values
})

comparison_1.index = range(1, 11)
pd.set_option("display.max_colwidth", None)
comparison_1

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44"
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Human Resources Professional Greater Boston Area 16,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7"
3,"Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212"
4,Human Resources Professional Greater Boston Area 16,Aspiring Human Resources Specialist Greater New York City Area 1,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1
5,"Human Resources Management Major Milpitas, California 18","Human Resources Management Major Milpitas, California 18",Director Human Resources at EY Greater Atlanta Area 349,"Seeking Human Resources Position Las Vegas, Nevada Area 48","Seeking Human Resources Position Las Vegas, Nevada Area 48","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500"
6,Human Resources Generalist at Schwan's Amerika Birlesik Devletleri 500,Director Human Resources at EY Greater Atlanta Area 349,"Human Resources Management Major Milpitas, California 18","Seeking employment opportunities within Customer Service or Patient Care Torrance, California 64","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103"
7,Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",Human Resources Professional Greater Boston Area 16,"Director Of Administration at

In [72]:
comparison_2 = pd.DataFrame({
    "BOW": df.nlargest(10,"similarity_2_bow")["all_data"].values,
    "TF-IDF": df.nlargest(10, "similarity_2_tfidf")["all_data"].values,
    "GloVe": df.nlargest(10, "similarity_2_glove")["all_data"].values,
    "BERT": df.nlargest(10, "similarity_2_bert")["all_data"].values,
    "SBERT": df.nlargest(10,"similarity_2_sbert")["all_data"].values,
    "E5": df.nlargest(10, "similarity_2_5e")["all_data"].values,
    "BGE": df.nlargest(10, "similarity_2_bge")["all_data"].values
})

comparison_2.index = range(1, 11)
pd.set_option("display.max_colwidth", None)
comparison_2

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Position Las Vegas, Nevada Area 48","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390"
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Human Resources Professional Greater Boston Area 16,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500
3,"Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7"
4,Human Resources Professional Greater Boston Area 16,Aspiring Human Resources Specialist Greater New York City Area 1,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Opportunities Chicago, Illinois 390",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Position Las Vegas, Nevada Area 48"
5,"Human Resources Management Major Milpitas, California 18","Human Resources Management Major Milpitas, California 18",Director Human Resources at EY Greater Atlanta Area 349,"Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44"
6,Human Resources Generalist at Schwan's Amerika Birlesik Devletleri 500,Director Human Resources at EY Greater Atlanta Area 349,"Human Resources Management Major Milpitas, California 18","Seeking employment opportunities within Customer Service or Patient Care Torrance, California 64","Human Resources Generalist at Loparex Raleigh-Durham, North Carolina Area 500",Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415
7,Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",Human Resources Professional Greater Boston Area 16,"Director Of Administration at Excellence Logging Katy, Texas 500","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,Aspiring Human Resources Specialist Greater New York City Area 1
8,Aspiring Human

In [74]:
star_candidate = X_e5[6] # student humber college aspiring human resource

starred_similarity = cosine_similarity(
    X_e5,
    star_candidate.reshape(1, -1)
).flatten()

alpha = 0.6

df['rerank_score'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["all_data", "rerank_score"]].sort_values(
    "rerank_score",
    ascending=False
).head(10)

,all_data,rerank_score
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,0.902248
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.868231
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.863316
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.862905
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.861159
9,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0.860888
99,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",0.858131
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.855061
93,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0.852855
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.848318


In [76]:
star_candidates = X_e5[2:6] # student humber college aspiring human resource
star_anchor = star_candidates.mean(axis=0)
starred_similarity = cosine_similarity(
    X_e5,
    star_anchor.reshape(1, -1)
).flatten()

alpha = 0.5

df['rerank_scores'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["job_title", "rerank_scores"]].sort_values(
    "rerank_scores",
    ascending=False
).head(10)

,job_title,rerank_scores
2,Aspiring Human Resources Professional,0.896573
5,Aspiring Human Resources Specialist,0.895933
98,Seeking Human Resources Position,0.889114
9,Seeking Human Resources HRIS and Generalist Positions,0.887591
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.886565
26,Aspiring Human Resources Management student seeking an internship,0.886372
27,Seeking Human Resources Opportunities,0.882969
75,Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment,0.877227
93,Seeking Human Resources Opportunities. Open to travel and relocation.,0.876549
88,Director Human Resources at EY,0.876324
